In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
CAND_DIR = BASE_PATH + "outputs_v2/candidates/"
OUTPUT_DIR = BASE_PATH + "outputs_v2/master/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("1. Khoi tao Spark Session cho Master Union...")
spark = SparkSession.builder \
    .appName("Master_Candidates_Union_Val") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

Mounted at /content/drive
1. Khoi tao Spark Session cho Master Union...


In [ ]:
print("2. Loading tat ca candidate files va Dong bo diem so...")
strategies = ["repurchase", "popularity", "sibling", "als", "itemcf", "categorical"]

# [NÂNG CẤP] Hàm tự động thêm cột điểm số nếu bảng đó không có
def standardize_schema(df):
    if "als_score" not in df.columns:
        df = df.withColumn("als_score", F.lit(0.0))
    if "itemcf_score" not in df.columns:
        df = df.withColumn("itemcf_score", F.lit(0.0))
    # Ép thứ tự cột chuẩn xác cho tất cả các bảng
    return df.select("customer_id", "article_id", "strategy", "als_score", "itemcf_score")

# Đọc tập TEST
test_dfs = []
for strat in strategies:
    path = f"{CAND_DIR}test_{strat}.parquet"
    if os.path.exists(path) or os.path.isdir(path):
        df = spark.read.parquet(path)
        test_dfs.append(standardize_schema(df))
        print(f" -> Loaded Test: {strat}")

# Đọc tập TRAIN
train_dfs = []
for strat in strategies:
    path = f"{CAND_DIR}train_{strat}.parquet"
    if os.path.exists(path) or os.path.isdir(path):
        df = spark.read.parquet(path)
        train_dfs.append(standardize_schema(df))
        print(f" -> Loaded Train: {strat}")

2. Loading tat ca candidate files va Dong bo diem so...
 -> Loaded Test: repurchase
 -> Loaded Test: popularity
 -> Loaded Test: sibling
 -> Loaded Test: als
 -> Loaded Test: itemcf
 -> Loaded Test: categorical
 -> Loaded Train: repurchase
 -> Loaded Train: popularity
 -> Loaded Train: sibling
 -> Loaded Train: als
 -> Loaded Train: itemcf
 -> Loaded Train: categorical


In [ ]:
def create_master_candidates(df_list):
    # 1. Nối dọc (Union) tất cả các bảng đã được đồng bộ cấu trúc
    master_df = df_list[0]
    for df in df_list[1:]:
        master_df = master_df.unionByName(df)

    # 2. Gom nhóm và [GIỮ LẠI ĐIỂM SỐ CAO NHẤT CỦA ALS & ITEMCF]
    master_dedup = master_df.groupBy("customer_id", "article_id") \
        .agg(
            F.collect_set("strategy").alias("sources"),
            F.max("als_score").alias("als_score"),          # Giữ lại điểm ALS
            F.max("itemcf_score").alias("itemcf_score")     # Giữ lại điểm ItemCF
        ) \
        .withColumn("source_count", F.size("sources"))

    return master_dedup

print("\n3. Merging Train candidates...")
train_master = create_master_candidates(train_dfs)
train_master.write.mode("overwrite").parquet(OUTPUT_DIR + "train_master_candidates.parquet")

print("4. Merging Test candidates...")
test_master = create_master_candidates(test_dfs)
test_master.write.mode("overwrite").parquet(OUTPUT_DIR + "test_master_candidates.parquet")


3. Merging Train candidates...
4. Merging Test candidates...


In [ ]:
print("5. Danh gia Master Candidates (Tren tap Test)...")
transactions = spark.read.parquet(INPUT_TRANS)
max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]
test_start = max_date - datetime.timedelta(days=7)

def evaluate_master(master_df, target_df, target_start, target_end):
    # Tính số ứng viên trung bình mỗi khách hàng
    avg_cands = master_df.groupBy("customer_id").count().select(F.avg("count")).collect()[0][0]

    # Tính Total Recall
    actuals = target_df.filter((F.col("t_dat_date") >= target_start) & (F.col("t_dat_date") < target_end)) \
        .select("customer_id", "article_id").dropDuplicates()

    total_actuals = actuals.count()
    hits = actuals.join(master_df, ["customer_id", "article_id"], "inner").dropDuplicates().count()
    recall = hits / total_actuals if total_actuals > 0 else 0

    print("-" * 50)
    print(f"SUMMARY MASTER CANDIDATES (TEST SET - VALIDATION)")
    print("-" * 50)
    print(f"Average Candidates per User: {avg_cands:.1f} items")
    print(f"Total Actual Purchases:      {total_actuals:,}")
    print(f"Total Hits Captured:         {hits:,}")
    print(f"TOTAL RECALL:                {recall:.4f} ({(recall*100):.2f}%)")
    print("-" * 50)

evaluate_master(test_master, transactions, test_start, max_date)

5. Danh gia Master Candidates (Tren tap Test)...
--------------------------------------------------
SUMMARY MASTER CANDIDATES (TEST SET - VALIDATION)
--------------------------------------------------
Average Candidates per User: 100.1 items
Total Actual Purchases:      207,996
Total Hits Captured:         20,147
TOTAL RECALL:                0.0969 (9.69%)
--------------------------------------------------
